In [ ]:
!pip install evaluate

In [ ]:
import os
import torch
import threading
import subprocess
import time
import numpy as np
import evaluate
from datasets import Dataset, DatasetDict, Image as DSImage
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    logging
)
from torch import nn

logging.set_verbosity_info()

MODEL_NAME       = "nvidia/mit-b3"
OUTPUT_DIR       = "/kaggle/working/checkpoints_magnifier"
FINAL_MODEL_DIR  = "/kaggle/working/final_magnifier"
DATASET_ROOT     = "/kaggle/input/magnified-rodent-data/magnified-rodent-data"
CHECKPOINT_INPUT = "/kaggle/input/magnifier-checkpoint"

TRAIN_IMG_DIR  = f"{DATASET_ROOT}/train/images"
TRAIN_MASK_DIR = f"{DATASET_ROOT}/train/masks"
VAL_IMG_DIR    = f"{DATASET_ROOT}/val/images"
VAL_MASK_DIR   = f"{DATASET_ROOT}/val/masks"

EPOCHS            = 30
LEARNING_RATE     = 3e-5
BATCH_SIZE        = 2
GRAD_ACCUMULATION = 8

def monitor_gpu(interval=60):
    while True:
        try:
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"]
            ).decode().strip().split('\n')
            stats = [f"GPU {i}: {line.split(',')[0]}% Util | {line.split(',')[1]}/{line.split(',')[2]} MB" for i, line in enumerate(result)]
            print(f"\n[GPU MONITOR] " + " | ".join(stats) + "\n")
        except Exception: pass
        time.sleep(interval)

class TimeoutCallback(TrainerCallback):
    def __init__(self, max_seconds=11 * 3600):
        self.start_time = time.time()
        self.max_seconds = max_seconds

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - self.start_time > self.max_seconds:
            print(f"\n[TIMEOUT] 11h elapsed — stopping and saving checkpoint.")
            control.should_save = True
            control.should_training_stop = True
        return control

def _load_split(img_dir, mask_dir, label):
    if not os.path.exists(img_dir):
        raise FileNotFoundError(f"CRITICAL: {img_dir} does not exist.")
    if not os.path.exists(mask_dir):
        raise FileNotFoundError(f"CRITICAL: {mask_dir} does not exist.")
    img_map  = {os.path.splitext(f)[0]: f for f in os.listdir(img_dir)  if f.endswith(('.jpg', '.png'))}
    mask_map = {os.path.splitext(f)[0]: f for f in os.listdir(mask_dir) if f.endswith(('.jpg', '.png'))}
    ids = sorted(set(img_map) & set(mask_map))
    print(f"  [{label}] {len(ids)} pairs")
    if not ids:
        raise ValueError(f"No matching image/mask pairs found in {img_dir}")
    ds = Dataset.from_dict({
        "image": [os.path.join(img_dir,  img_map[i]) for i in ids],
        "label": [os.path.join(mask_dir, mask_map[i]) for i in ids],
    })
    ds = ds.cast_column("image", DSImage())
    ds = ds.cast_column("label", DSImage())
    return ds

def load_dataset():
    print("--- LOADING DATA ---")
    return DatasetDict({
        "train": _load_split(TRAIN_IMG_DIR, TRAIN_MASK_DIR, "train"),
        "test":  _load_split(VAL_IMG_DIR,   VAL_MASK_DIR,   "val"),
    })

processor = SegformerImageProcessor.from_pretrained(
    MODEL_NAME, do_resize=True, size={"height": 512, "width": 512}
)

def train_transforms(example_batch):
    images = [x.convert("RGB") for x in example_batch["image"]]
    labels = []
    for x in example_batch["label"]:
        mask_np = np.array(x.convert("L"))
        mask_np = np.where(mask_np > 0, 1, 0).astype(np.uint8)
        labels.append(mask_np)
    return processor(images, labels, return_tensors="pt")

metric = evaluate.load("mean_iou")

def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)
        logits_tensor = nn.functional.interpolate(
            logits_tensor, size=labels.shape[-2:], mode="bilinear", align_corners=False
        ).argmax(dim=1)
        metrics = metric.compute(
            predictions=logits_tensor.numpy(),
            references=labels,
            num_labels=2,
            ignore_index=255,
            reduce_labels=False,
        )
        result = {k: v.tolist() if isinstance(v, np.ndarray) else v for k, v in metrics.items()}
        print(f"[EVAL] IoU={result.get('mean_iou', '?'):.4f}")
        return result

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")
        upsampled_logits = nn.functional.interpolate(
            logits, size=labels.shape[-2:], mode="bilinear", align_corners=False
        )
        weights  = torch.tensor([1.0, 5.0]).to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss     = loss_fct(upsampled_logits, labels)
        return (loss, outputs) if return_outputs else loss

def main():
    ds = load_dataset()
    ds["train"].set_transform(train_transforms)
    ds["test"].set_transform(train_transforms)

    model = SegformerForSemanticSegmentation.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label={0: "background", 1: "rat"},
        label2id={"background": 0, "rat": 1},
        ignore_mismatched_sizes=True
    )

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION,
        fp16=True,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="mean_iou",
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=ds["train"],
        eval_dataset=ds["test"],
        compute_metrics=compute_metrics,
        callbacks=[TimeoutCallback(), EarlyStoppingCallback(early_stopping_patience=8)],
    )

    last_checkpoint = None
    if os.path.isdir(OUTPUT_DIR):
        ckpts = sorted(
            [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")],
            key=lambda x: int(x.split("-")[1])
        )
        if ckpts:
            last_checkpoint = os.path.join(OUTPUT_DIR, ckpts[-1])
            print(f"!!! RESUMING FROM WORKING DIR: {last_checkpoint} !!!")

    if last_checkpoint is None and os.path.isdir(CHECKPOINT_INPUT):
        ckpts = sorted(
            [d for d in os.listdir(CHECKPOINT_INPUT) if d.startswith("checkpoint-")],
            key=lambda x: int(x.split("-")[1])
        )
        if ckpts:
            last_checkpoint = os.path.join(CHECKPOINT_INPUT, ckpts[-1])
            print(f"!!! RESUMING FROM INPUT DATASET: {last_checkpoint} !!!")

    print("--- TRAINING START: MAGNIFIER (BCE) ---")
    trainer.train(resume_from_checkpoint=last_checkpoint)

    print(f"--- SAVING TO {FINAL_MODEL_DIR} ---")
    trainer.save_model(FINAL_MODEL_DIR)
    processor.save_pretrained(FINAL_MODEL_DIR)
    print("DONE.")

    print("Shutting down kernel to preserve GPU hours.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(False)

threading.Thread(target=monitor_gpu, daemon=True).start()
main()